# Mô phỏng & Đánh giá Thuật toán GMM-HMM Phân cấp (Xie et al. 2025)
Mô hình GMM-HMM phân cấp dựa trên SSE phân loại 3 trạng thái ánh mắt: Fixation, Smooth Pursuit, Saccade.


In [1]:
# %pip install -q hmmlearn scikit-learn matplotlib pandas numpy scipy


In [2]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from hmmlearn.hmm import GMMHMM
import warnings
import logging
from itertools import product
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

FIX, PUR, SAC = 0, 1, 2

BASE_CFG = dict(
    fs=90,
    seed=42,
    K_MAX=9,
    KMEANS_MAX_ITER=30,
    KMEANS_RESTARTS=3,
    L2_FEATURES=['v'],
    N_ITER=25,
    TOL=1e-4,
    N_RESTARTS=2
)

def kmeans_sse(X, k, max_iter, restarts, rng):
    best = np.inf
    for _ in range(restarts):
        mu = X[rng.choice(len(X), k, replace=False)]
        for _ in range(max_iter):
            d = ((X[:, None, :] - mu[None]) ** 2).sum(-1)
            c = d.argmin(1)
            new = np.array([X[c == i].mean(0) if (c == i).any() else mu[i] for i in range(k)])
            if np.allclose(new, mu): break
            mu = new
        sse = sum(((X[c == i] - mu[i]) ** 2).sum() for i in range(k))
        best = min(best, sse)
    return best

def find_elbow(ks, sse):
    x = (ks - ks.min()) / (ks.max() - ks.min())
    y = (sse - sse.min()) / (sse.max() - sse.min() + 1e-12)
    d = np.abs((y[-1] - y[0]) * x - (x[-1] - x[0]) * y + x[-1] * y[0] - y[-1] * x[0])
    return int(ks[d.argmax()])

def step0_optimal_k(df, cfg):
    rng = np.random.default_rng(cfg['seed'])
    X = df[['x', 'y']].values
    ks = np.arange(1, min(cfg['K_MAX'], len(X) - 1) + 1)
    sse = np.array([kmeans_sse(X, k, cfg['KMEANS_MAX_ITER'], cfg['KMEANS_RESTARTS'], rng) for k in ks])
    return find_elbow(ks, sse), ks, sse

def fit_gmmhmm(X, n_states, cfg, seed=0):
    for n_mix in dict.fromkeys([cfg['N_MIX'], 1]):
        best = None
        for r in range(cfg['N_RESTARTS']):
            try:
                m = GMMHMM(n_components=n_states, n_mix=n_mix, covariance_type=cfg['COV_TYPE'],
                           n_iter=cfg['N_ITER'], tol=cfg['TOL'], min_covar=1e-3, random_state=seed + r)
                m.fit(X)
                s = m.score(X)
                if np.isfinite(s) and (best is None or s > best[1]): best = (m, s)
            except Exception:
                continue
        if best is not None:
            return best[0]
    return None

def viterbi_states(model, X):
    _, states = model.decode(X, algorithm="viterbi")
    return states

def _scale(X, on):
    return StandardScaler().fit_transform(X) if on else X

def hierarchical_gmm_hmm(df, cfg, k=None):
    N = len(df)
    if k is None:
        k, ks_, sse_ = step0_optimal_k(df, cfg)
    X1 = _scale(df[cfg['L1_FEATURES']].values, cfg['STANDARDIZE'])
    m1 = fit_gmmhmm(X1, k, cfg, cfg['seed'])
    if m1 is None: return None
    seg_id = viterbi_states(m1, X1)
    
    v = df['v'].values
    local = np.full(N, -1)
    seg_stats = []
    tiny = []
    for s in np.unique(seg_id):
        idx = np.where(seg_id == s)[0]
        if len(idx) < cfg['MIN_SEG_LEN']:
            tiny.append(idx); continue
        X2 = _scale(df.loc[idx, cfg['L2_FEATURES']].values, cfg['STANDARDIZE'])
        m2 = fit_gmmhmm(X2, 3, cfg, cfg['seed'] + 100)
        if m2 is None:
            tiny.append(idx); continue
        st = viterbi_states(m2, X2)
        local[idx] = st
        for c in np.unique(st):
            seg_stats.append((s, c, v[idx][st == c].mean()))
            
    if not seg_stats: return None
    lv = np.log10(np.array([m for _, _, m in seg_stats]) + 1e-6).reshape(-1, 1)
    km = KMeans(3, n_init=10, random_state=cfg['seed']).fit(lv)
    order = np.argsort(km.cluster_centers_.ravel())
    rank = {int(c): r for r, c in enumerate(order)}
    centers = km.cluster_centers_.ravel()[order]
    state2label = {(s, c): rank[int(g)] for (s, c, _), g in zip(seg_stats, km.labels_)}
    
    pred = np.full(N, -1)
    for i in range(N):
        if local[i] >= 0:
            pred[i] = state2label.get((seg_id[i], local[i]), 0)
    for idx in tiny:
        d = np.abs(np.log10(v[idx] + 1e-6)[:, None] - centers[None])
        pred[idx] = d.argmin(1)
    return dict(k=k, seg_id=seg_id, pred=pred, centers=10 ** centers, n_tiny=len(tiny))

def evaluate_one_file(file_item, cfg):
    fname, df_trial, y_true, k_value = file_item
    try:
        out = hierarchical_gmm_hmm(df_trial, cfg, k=k_value)
        if out is None:
            return None
        y_pred = out['pred']
        if len(y_true) != len(y_pred):
            print(f'Bỏ qua {fname}: số nhãn ({len(y_true)}) khác số dự đoán ({len(y_pred)})')
            return None
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[FIX, PUR, SAC], zero_division=0
        )
        accuracy_by_label = np.array([
            accuracy_score(y_true == label, y_pred == label)
            for label in [FIX, PUR, SAC]
        ])
        return {
            'accuracy': acc,
            'f1_macro': f1.mean(),
            'precision_macro': prec.mean(),
            'recall_macro': rec.mean(),
            'precision_by_label': prec,
            'recall_by_label': rec,
            'f1_by_label': f1,
            'accuracy_by_label': accuracy_by_label,
        }
    except Exception as exc:
        print(f'Lỗi ở {fname} với cấu hình {cfg}: {exc}')
        return None

# Hyperparameter Grid
grid = {
    'L1_FEATURES': [['x', 'y'], ['x', 'y', 'v']],
    'COV_TYPE': ['full', 'diag', 'spherical'],
    'N_MIX': [1, 2, 3],
    'MIN_SEG_LEN': [10, 15, 20],
    'STANDARDIZE': [True, False]
}

param_combinations = [dict(zip(grid.keys(), v)) for v in product(*grid.values())]

# Load files relative to this notebook/project instead of a machine-specific path
BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if os.path.basename(BASE_DIR).lower() != 'eyemovement':
    BASE_DIR = os.path.join(BASE_DIR, 'eyemovement')
data_dir = os.path.join(BASE_DIR, 'dataset', 'testdataset')
res_dir = os.path.join(BASE_DIR, 'dataset', 'results')
txt_files = sorted(glob.glob(os.path.join(data_dir, '*.txt')))
if not txt_files:
    raise FileNotFoundError(f'Không tìm thấy file .txt trong: {data_dir}')

# Run the grid search on every file in the dataset
eval_files = txt_files

print(f"Bắt đầu Grid Search trên {len(param_combinations)} tổ hợp tham số x {len(eval_files)} files...")

# Load and validate each input/result pair once.
file_cache = []
for txt_path in eval_files:
    fname = os.path.basename(txt_path)
    base_name = os.path.splitext(fname)[0]
    csv_path = os.path.join(res_dir, f"{base_name}_results.csv")
    if not os.path.exists(csv_path):
        print(f'Bỏ qua {fname}: không tìm thấy {os.path.basename(csv_path)}')
        continue
    arr = np.loadtxt(txt_path)
    csv_df = pd.read_csv(csv_path)
    if arr.ndim != 2 or arr.shape[1] < 3:
        print(f'Bỏ qua {fname}: dữ liệu phải có ít nhất 3 cột')
        continue
    y_true = csv_df['manually results'].values.astype(int)
    if len(arr) != len(y_true):
        print(f'Bỏ qua {fname}: số dòng dữ liệu ({len(arr)}) khác số nhãn ({len(y_true)})')
        continue
    df_trial = pd.DataFrame({
        't': np.arange(len(arr)) / BASE_CFG['fs'],
        'x': arr[:, 0], 'y': arr[:, 1], 'v': arr[:, 2]
    })
    k_value, _, _ = step0_optimal_k(df_trial, BASE_CFG)
    file_cache.append((fname, df_trial, y_true, k_value))

print(f'Đã nạp và tiền xử lý {len(file_cache)}/{len(eval_files)} files.')

grid_results = []

for idx, p in enumerate(param_combinations, 1):
    cfg = {**BASE_CFG, **p}
    
    acc_list, f1_list, prec_list, rec_list = [], [], [], []
    precision_by_label, recall_by_label, f1_by_label, accuracy_by_label = [], [], [], []
    
    workers = max(1, min(8, os.cpu_count() or 1, len(file_cache)))
    metrics = Parallel(n_jobs=workers, backend='loky', verbose=0)(
        delayed(evaluate_one_file)(file_item, cfg)
        for file_item in file_cache
    )
    for metric in metrics:
        if metric is None:
            continue
        acc_list.append(metric['accuracy'])
        f1_list.append(metric['f1_macro'])
        prec_list.append(metric['precision_macro'])
        rec_list.append(metric['recall_macro'])
        precision_by_label.append(metric['precision_by_label'])
        recall_by_label.append(metric['recall_by_label'])
        f1_by_label.append(metric['f1_by_label'])
        accuracy_by_label.append(metric['accuracy_by_label'])
            
    if acc_list:
        mean_acc = np.mean(acc_list)
        std_acc = np.std(acc_list)
        mean_f1 = np.mean(f1_list)
        mean_prec = np.mean(prec_list)
        mean_rec = np.mean(rec_list)
        label_precision = np.mean(precision_by_label, axis=0)
        label_recall = np.mean(recall_by_label, axis=0)
        label_f1 = np.mean(f1_by_label, axis=0)
        label_accuracy = np.mean(accuracy_by_label, axis=0)
        
        grid_results.append({
            'L1_FEATURES': str(p['L1_FEATURES']),
            'COV_TYPE': p['COV_TYPE'],
            'N_MIX': p['N_MIX'],
            'MIN_SEG_LEN': p['MIN_SEG_LEN'],
            'STANDARDIZE': p['STANDARDIZE'],
            'Mean_Accuracy': mean_acc,
            'Std_Accuracy': std_acc,
            'Mean_F1': mean_f1,
            'Mean_Precision': mean_prec,
            'Mean_Recall': mean_rec,
            'Precision_Fixation': label_precision[0],
            'Recall_Fixation': label_recall[0],
            'F1_Fixation': label_f1[0],
            'Accuracy_Fixation': label_accuracy[0],
            'Precision_Smooth_Pursuit': label_precision[1],
            'Recall_Smooth_Pursuit': label_recall[1],
            'F1_Smooth_Pursuit': label_f1[1],
            'Accuracy_Smooth_Pursuit': label_accuracy[1],
            'Precision_Saccade': label_precision[2],
            'Recall_Saccade': label_recall[2],
            'F1_Saccade': label_f1[2],
            'Accuracy_Saccade': label_accuracy[2]
        })

if not grid_results:
    raise RuntimeError('Không có cấu hình nào chạy thành công. Xem các lỗi chi tiết ở trên.')
df_grid = pd.DataFrame(grid_results).sort_values(by='Mean_Accuracy', ascending=False)
print("\n=== TOP 10 TỔ HỢP SIÊU THAM SỐ TỐI ƯU NHẤT ===")
print(df_grid.head(10).to_string(index=False))

df_grid.to_csv(os.path.join(BASE_DIR, 'grid_search_results.csv'), index=False)
print("\nĐã lưu toàn bộ kết quả Grid Search vào grid_search_results.csv!")


Bắt đầu Grid Search trên 108 tổ hợp tham số x 87 files...
Đã nạp và tiền xử lý 87/87 files.

=== TOP 10 TỔ HỢP SIÊU THAM SỐ TỐI ƯU NHẤT ===
    L1_FEATURES  COV_TYPE  N_MIX  MIN_SEG_LEN  STANDARDIZE  Mean_Accuracy  Std_Accuracy  Mean_F1  Mean_Precision  Mean_Recall  Precision_Fixation  Recall_Fixation  F1_Fixation  Accuracy_Fixation  Precision_Smooth_Pursuit  Recall_Smooth_Pursuit  F1_Smooth_Pursuit  Accuracy_Smooth_Pursuit  Precision_Saccade  Recall_Saccade  F1_Saccade  Accuracy_Saccade
['x', 'y', 'v'] spherical      1           20        False       0.874477      0.080030 0.835521        0.856385     0.847193            0.942700         0.923802     0.927659           0.914570                  0.738499               0.803982           0.756343                 0.874578           0.887956        0.813796    0.822561          0.959806
['x', 'y', 'v'] spherical      1           10        False       0.874477      0.080030 0.835521        0.856385     0.847193            0.942700         